# ☁️ Cloud Platform Cost Ingestion

Fetches billing data from Azure / AWS / GCP and writes to a unified Delta table:
`<catalog>.platform.cloud_platform_costs`

**Run daily** (schedule via Databricks Workflows).

| Parameter | Description |
|-----------|-------------|
| `cloud_provider` | `azure` \| `aws` \| `gcp` |
| `lookback_days` | Days to backfill (default `2`, covers API lag) |
| `catalog` | UC catalog to write to (default `workspace`) |

In [ ]:
# ── Parameters (override via Databricks Widgets or Workflows job parameters) ──
import os

try:
    dbutils.widgets.text('cloud_provider', 'azure',  'Cloud Provider (azure|aws|gcp)')
    dbutils.widgets.text('lookback_days',  '2',      'Lookback days')
    dbutils.widgets.text('catalog',        'workspace', 'UC Catalog')
    CLOUD    = dbutils.widgets.get('cloud_provider').strip().lower()
    LOOKBACK = int(dbutils.widgets.get('lookback_days'))
    CATALOG  = dbutils.widgets.get('catalog').strip()
except Exception:
    CLOUD    = os.environ.get('CLOUD_PROVIDER', 'azure').lower()
    LOOKBACK = int(os.environ.get('LOOKBACK_DAYS', '2'))
    CATALOG  = os.environ.get('UC_CATALOG', 'workspace')

assert CLOUD in ('azure', 'aws', 'gcp'), f'Unsupported cloud_provider: {CLOUD}'
print(f'Provider: {CLOUD}  |  Lookback: {LOOKBACK}d  |  Catalog: {CATALOG}')

In [ ]:
# ── Common helpers ────────────────────────────────────────────────────────────
import time
from datetime import datetime, timedelta, date
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    HOST  = ctx.apiUrl().get()
    TOKEN = ctx.apiToken().get()
except Exception:
    HOST  = os.environ.get('DATABRICKS_HOST', '')
    TOKEN = os.environ.get('DATABRICKS_TOKEN', '')

WAREHOUSE_ID = os.environ.get('DATABRICKS_WAREHOUSE_ID', '')
client = WorkspaceClient(host=HOST, token=TOKEN)

def run_sql(label, sql, timeout_sec=120):
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout='50s')
    deadline = time.time() + timeout_sec
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() > deadline:
            print(f'  TIMEOUT: {label}'); return False
        time.sleep(3)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state == StatementState.SUCCEEDED:
        print(f'  OK: {label}'); return True
    err = getattr(resp.status.error, 'message', str(resp.status.state))
    print(f'  WARN ({err[:120]}): {label}'); return False

TARGET_TABLE = f'`{CATALOG}`.platform.cloud_platform_costs'

END_DATE   = date.today() - timedelta(days=1)          # yesterday (API lag)
START_DATE = END_DATE - timedelta(days=LOOKBACK - 1)
print(f'Date window: {START_DATE} → {END_DATE}')

In [ ]:
# -- Ensure schema + table exist ---------------------------------------------
run_sql('create schema', f'CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.platform')
run_sql('create table', f'''
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    cloud_provider   STRING        NOT NULL COMMENT 'azure | aws | gcp',
    account_id       STRING        NOT NULL COMMENT 'Subscription / Account / Project ID',
    account_name     STRING                 COMMENT 'Human-readable name',
    usage_date       DATE          NOT NULL,
    service_name     STRING                 COMMENT 'Top-level service (e.g. Amazon EC2)',
    service_category STRING                 COMMENT 'Compute | Storage | Database | Analytics | AI / ML | ...',
    sku_description  STRING                 COMMENT 'Meter/SKU (e.g. BoxUsage:m5.xlarge)',
    usage_unit       STRING                 COMMENT 'Unit of measure (Hrs, GB-Mo, 1K Tokens)',
    usage_quantity   DOUBLE                 COMMENT 'Quantity consumed',
    unit_price       DOUBLE                 COMMENT 'Price per unit after reserved/spot discount',
    cost_usd         DOUBLE                 COMMENT 'usage_quantity * unit_price',
    currency         STRING,
    pricing_model    STRING                 COMMENT 'ON_DEMAND | RESERVED_1Y | RESERVED_3Y | SPOT | COMMITTED_1Y',
    region           STRING,
    resource_group   STRING                 COMMENT 'Azure resource group; empty for AWS/GCP',
    tags             MAP<STRING, STRING>    COMMENT 'team, environment, cost_center, project',
    ingested_at      TIMESTAMP
)
USING DELTA
PARTITIONED BY (cloud_provider, usage_date)
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
''')
print('Table ready:', TARGET_TABLE)


In [ ]:
# -- Azure Cost Management ---------------------------------------------------
def fetch_azure(start: date, end: date) -> list[dict]:
    import json, requests
    tenant_id     = os.environ.get('AZURE_TENANT_ID', '')
    client_id     = os.environ.get('AZURE_CLIENT_ID', '')
    client_secret = os.environ.get('AZURE_CLIENT_SECRET', '')
    sub_ids_raw   = os.environ.get('AZURE_SUBSCRIPTION_IDS', '')
    sub_ids       = [s.strip() for s in sub_ids_raw.split(',') if s.strip()]

    if not all([tenant_id, client_id, client_secret, sub_ids]):
        print('  SKIP Azure: missing credentials'); return []

    token_resp = requests.post(
        f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token',
        data={'grant_type':'client_credentials','client_id':client_id,
              'client_secret':client_secret,'scope':'https://management.azure.com/.default'},
        timeout=30,
    )
    token_resp.raise_for_status()
    bearer  = token_resp.json()['access_token']
    headers = {'Authorization': f'Bearer {bearer}', 'Content-Type': 'application/json'}
    rows    = []
    now_ts  = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

    for sub_id in sub_ids:
        url = (
            f'https://management.azure.com/subscriptions/{sub_id}'
            f'/providers/Microsoft.CostManagement/query?api-version=2023-11-01'
        )
        body = {
            'type': 'ActualCost',
            'timeframe': 'Custom',
            'timePeriod': {'from': start.isoformat(), 'to': end.isoformat()},
            'dataset': {
                'granularity': 'Daily',
                'aggregation': {
                    'totalCost':     {'name': 'Cost',          'function': 'Sum'},
                    'totalQuantity': {'name': 'UsageQuantity', 'function': 'Sum'},
                },
                'grouping': [
                    {'type': 'Dimension', 'name': 'ServiceName'},
                    {'type': 'Dimension', 'name': 'ServiceCategory'},
                    {'type': 'Dimension', 'name': 'MeterName'},
                    {'type': 'Dimension', 'name': 'UnitOfMeasure'},
                    {'type': 'Dimension', 'name': 'ResourceLocation'},
                    {'type': 'Dimension', 'name': 'ResourceGroupName'},
                ],
            },
        }
        resp = requests.post(url, headers=headers, json=body, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        cols = [c['name'] for c in data['properties']['columns']]
        for row in data['properties']['rows']:
            rec  = dict(zip(cols, row))
            cost = float(rec.get('Cost', 0) or 0)
            qty  = float(rec.get('UsageQuantity', 0) or 0)
            up   = round(cost / qty, 8) if qty else 0.0
            rows.append({
                'cloud_provider':   'azure',
                'account_id':       sub_id,
                'account_name':     sub_id,
                'usage_date':       str(rec.get('UsageDate', ''))[:10],
                'service_name':     rec.get('ServiceName', ''),
                'service_category': rec.get('ServiceCategory', ''),
                'sku_description':  rec.get('MeterName', ''),
                'usage_unit':       rec.get('UnitOfMeasure', ''),
                'usage_quantity':   qty,
                'unit_price':       up,
                'cost_usd':         cost,
                'currency':         rec.get('Currency', 'USD'),
                'pricing_model':    'ON_DEMAND',
                'region':           rec.get('ResourceLocation', ''),
                'resource_group':   rec.get('ResourceGroupName', ''),
                'tags':             {},
                'ingested_at':      now_ts,
            })
    print(f'  Azure rows fetched: {len(rows)}')
    return rows


In [ ]:
# -- AWS Cost Explorer -------------------------------------------------------
def fetch_aws(start: date, end: date) -> list[dict]:
    try:
        import boto3
    except ImportError:
        print('  SKIP AWS: boto3 not installed. Run: pip install boto3'); return []

    key_id     = os.environ.get('AWS_ACCESS_KEY_ID', '')
    secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY', '')
    account_ids_raw = os.environ.get('AWS_ACCOUNT_IDS', '')
    account_ids     = [a.strip() for a in account_ids_raw.split(',') if a.strip()]

    if not all([key_id, secret_key]):
        print('  SKIP AWS: missing credentials'); return []

    _AWS_CAT = {
        'Amazon Elastic Compute Cloud': 'Compute',
        'AWS Lambda': 'Serverless',
        'Amazon Simple Storage Service': 'Storage',
        'Amazon Elastic Block Store': 'Storage',
        'Amazon Relational Database Service': 'Database',
        'Amazon DynamoDB': 'Database',
        'Amazon Redshift': 'Analytics',
        'Amazon Virtual Private Cloud': 'Networking',
        'AWS Data Transfer': 'Networking',
        'Amazon CloudFront': 'Networking',
        'Amazon SageMaker': 'AI / ML',
        'Amazon Bedrock': 'AI / ML',
    }

    ce = boto3.client('ce', region_name='us-east-1',
                      aws_access_key_id=key_id, aws_secret_access_key=secret_key)

    resp = ce.get_cost_and_usage(
        TimePeriod={'Start': start.isoformat(),
                    'End': (end + timedelta(days=1)).isoformat()},
        Granularity='DAILY',
        Metrics=['UnblendedCost', 'UsageQuantity'],
        GroupBy=[
            {'Type': 'DIMENSION', 'Key': 'SERVICE'},
            {'Type': 'DIMENSION', 'Key': 'LINKED_ACCOUNT'},
            {'Type': 'DIMENSION', 'Key': 'REGION'},
            {'Type': 'DIMENSION', 'Key': 'USAGE_TYPE'},
        ],
    )

    rows   = []
    now_ts = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

    for period in resp.get('ResultsByTime', []):
        usage_date = period['TimePeriod']['Start']
        for group in period.get('Groups', []):
            keys     = group['Keys']
            svc      = keys[0] if len(keys) > 0 else ''
            acct_id  = keys[1] if len(keys) > 1 else ''
            region   = keys[2] if len(keys) > 2 else ''
            usage_tp = keys[3] if len(keys) > 3 else ''
            cost     = float(group['Metrics']['UnblendedCost']['Amount'])
            qty      = float(group['Metrics']['UsageQuantity']['Amount'])
            currency = group['Metrics']['UnblendedCost']['Unit']
            up       = round(cost / qty, 8) if qty else 0.0
            ut_lower = usage_tp.lower()
            if 'spot' in ut_lower:
                pm = 'SPOT'
            elif 'heavyusage' in ut_lower or 'partialupfront' in ut_lower:
                pm = 'RESERVED_1Y'
            else:
                pm = 'ON_DEMAND'
            if account_ids and acct_id not in account_ids:
                continue
            rows.append({
                'cloud_provider':   'aws',
                'account_id':       acct_id,
                'account_name':     acct_id,
                'usage_date':       usage_date,
                'service_name':     svc,
                'service_category': _AWS_CAT.get(svc, 'Other'),
                'sku_description':  usage_tp,
                'usage_unit':       'Hrs',
                'usage_quantity':   qty,
                'unit_price':       up,
                'cost_usd':         cost,
                'currency':         currency,
                'pricing_model':    pm,
                'region':           region,
                'resource_group':   '',
                'tags':             {},
                'ingested_at':      now_ts,
            })
    print(f'  AWS rows fetched: {len(rows)}')
    return rows


In [ ]:
# -- GCP Cloud Billing -------------------------------------------------------
def fetch_gcp(start: date, end: date) -> list[dict]:
    try:
        from google.oauth2 import service_account
        from google.cloud import bigquery
        import json as _json
    except ImportError:
        print('  SKIP GCP: google-cloud-bigquery not installed.')
        print('  Run: pip install google-cloud-bigquery google-auth')
        return []

    sa_json         = os.environ.get('GCP_SERVICE_ACCOUNT_JSON', '')
    bq_table        = os.environ.get('GCP_BILLING_BQ_TABLE', '')
    project_ids_raw = os.environ.get('GCP_PROJECT_IDS', '')
    project_ids     = [p.strip() for p in project_ids_raw.split(',') if p.strip()]

    if not all([sa_json, bq_table]):
        print('  SKIP GCP: missing GCP_SERVICE_ACCOUNT_JSON or GCP_BILLING_BQ_TABLE')
        return []

    creds = service_account.Credentials.from_service_account_info(
        _json.loads(sa_json),
        scopes=['https://www.googleapis.com/auth/bigquery.readonly'],
    )
    bq = bigquery.Client(credentials=creds, project=creds.project_id)

    project_filter = ''
    if project_ids:
        quoted = ', '.join(f"'{p}'" for p in project_ids)
        project_filter = f'AND project.id IN ({quoted})'

    _GCP_CAT = {
        'Compute Engine': 'Compute', 'Cloud Run': 'Serverless',
        'Google Kubernetes Engine': 'Compute', 'Cloud Storage': 'Storage',
        'Persistent Disk': 'Storage', 'Cloud SQL': 'Database',
        'BigQuery': 'Analytics', 'Dataproc': 'Analytics',
        'Vertex AI': 'AI / ML', 'Cloud Networking': 'Networking',
        'Cloud CDN': 'Networking',
    }

    query = f"""
        SELECT
            project.id                               AS project_id,
            project.name                             AS project_name,
            DATE(usage_start_time)                   AS usage_date,
            service.description                      AS service_name,
            sku.description                          AS sku_description,
            usage.pricing_unit                       AS usage_unit,
            SUM(usage.amount)                        AS usage_quantity,
            COALESCE(location.region,
                     location.location, 'global')    AS region,
            SUM(cost) + SUM(IFNULL(
                (SELECT SUM(c.amount) FROM UNNEST(credits) c), 0
            ))                                       AS cost_usd,
            currency
        FROM `{bq_table}`
        WHERE DATE(usage_start_time) BETWEEN '{start}' AND '{end}'
        {project_filter}
        GROUP BY 1,2,3,4,5,6,8,10
        ORDER BY usage_date, cost_usd DESC
    """

    rows   = []
    now_ts = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

    for row in bq.query(query):
        svc  = row['service_name'] or ''
        qty  = float(row['usage_quantity'] or 0)
        cost = float(row['cost_usd'] or 0)
        up   = round(cost / qty, 8) if qty else 0.0
        rows.append({
            'cloud_provider':   'gcp',
            'account_id':       row['project_id'] or '',
            'account_name':     row['project_name'] or row['project_id'] or '',
            'usage_date':       str(row['usage_date']),
            'service_name':     svc,
            'service_category': _GCP_CAT.get(svc, 'Other'),
            'sku_description':  row['sku_description'] or '',
            'usage_unit':       row['usage_unit'] or '',
            'usage_quantity':   qty,
            'unit_price':       up,
            'cost_usd':         cost,
            'currency':         row['currency'] or 'USD',
            'pricing_model':    'ON_DEMAND',
            'region':           row['region'] or '',
            'resource_group':   '',
            'tags':             {},
            'ingested_at':      now_ts,
        })
    print(f'  GCP rows fetched: {len(rows)}')
    return rows


In [ ]:
# ── Fetch data for selected provider ─────────────────────────────────────────
fetch_fn = {'azure': fetch_azure, 'aws': fetch_aws, 'gcp': fetch_gcp}[CLOUD]
rows = fetch_fn(START_DATE, END_DATE)

if not rows:
    print('No rows to ingest — exiting.')
else:
    print(f'Fetched {len(rows)} rows for {CLOUD}')

In [ ]:
# -- Write to Delta (MERGE to avoid duplicates) ------------------------------
if rows:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import (
        StructType, StructField, StringType, DateType,
        DoubleType, TimestampType, MapType
    )
    from pyspark.sql.functions import lit, to_date, to_timestamp
    from delta.tables import DeltaTable

    spark = SparkSession.builder.getOrCreate()

    schema = StructType([
        StructField('cloud_provider',   StringType(), False),
        StructField('account_id',       StringType(), False),
        StructField('account_name',     StringType(), True),
        StructField('usage_date',       StringType(), False),
        StructField('service_name',     StringType(), True),
        StructField('service_category', StringType(), True),
        StructField('sku_description',  StringType(), True),
        StructField('usage_unit',       StringType(), True),
        StructField('usage_quantity',   DoubleType(), True),
        StructField('unit_price',       DoubleType(), True),
        StructField('cost_usd',         DoubleType(), True),
        StructField('currency',         StringType(), True),
        StructField('pricing_model',    StringType(), True),
        StructField('region',           StringType(), True),
        StructField('resource_group',   StringType(), True),
        StructField('ingested_at',      StringType(), True),
    ])

    plain_rows = [{k: v for k, v in r.items() if k != 'tags'} for r in rows]
    df = (
        spark.createDataFrame(plain_rows, schema=schema)
        .withColumn('usage_date',  to_date('usage_date'))
        .withColumn('ingested_at', to_timestamp('ingested_at'))
        .withColumn('tags', lit(None).cast(MapType(StringType(), StringType())))
    )

    full_table = f'{CATALOG}.platform.cloud_platform_costs'

    (
        DeltaTable.forName(spark, full_table).alias('t')
        .merge(
            df.alias('s'),
            'T.cloud_provider  = S.cloud_provider '
            'AND T.account_id      = S.account_id '
            'AND T.usage_date      = S.usage_date '
            'AND T.service_name    = S.service_name '
            'AND T.sku_description = S.sku_description '
            'AND T.region          = S.region '
            'AND T.resource_group  = S.resource_group'
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f'MERGE complete --> {full_table}')
    spark.sql(
        f'SELECT cloud_provider, COUNT(*) AS rows, '
        f'ROUND(SUM(cost_usd),0) AS total_usd '
        f'FROM {full_table} GROUP BY 1 ORDER BY 1'
    ).show()


## Setup Notes

### Azure
1. Create a Service Principal: `az ad sp create-for-rbac --name databricks-cost-reader`
2. Assign **Cost Management Reader** role on each subscription
3. Store secrets in Databricks Secrets or env vars:
   - `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_SUBSCRIPTION_IDS`

### AWS
1. Create an IAM user/role with policy: `ce:GetCostAndUsage`
2. Install boto3 on the cluster: `pip install boto3`
3. Store: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_ACCOUNT_IDS`

### GCP
1. Enable [Cloud Billing export to BigQuery](https://cloud.google.com/billing/docs/how-to/export-data-bigquery)
2. Create a Service Account with **BigQuery Data Viewer** on the billing dataset
3. Install: `pip install google-cloud-bigquery google-auth`
4. Store: `GCP_SERVICE_ACCOUNT_JSON` (full JSON), `GCP_BILLING_BQ_TABLE`, `GCP_PROJECT_IDS`

### Schedule (Databricks Workflows)
Create a job with three tasks (one per provider), each passing `cloud_provider` as a widget parameter, scheduled daily at 06:00 UTC.